In [1]:
import pandas as pd
import sqlite3

In [2]:
# conectamos con la base de datos my_database.db
connection = sqlite3.connect("database_cervezas.db")

# obtenemos un cursor que utilizamos para las queries
crsr = connection.cursor()

In [3]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)

In [4]:
res = crsr.execute("SELECT name FROM sqlite_master WHERE type='table'")
for name in res:
    print(name[0])

CERVEZAS
BARES
EMPLEADOS
REPARTO


In [5]:
# crea las tablas

In [6]:
query = '''
drop table CERVEZAS
'''

crsr.execute(query)


In [7]:
query = '''
drop table reparto
'''

crsr.execute(query)

In [8]:
query = '''
drop table bares
'''

crsr.execute(query)

In [9]:
query = '''
drop table empleados
'''

crsr.execute(query)

In [10]:
query = '''
CREATE TABLE IF NOT EXISTS CERVEZAS (
    CodC int,
    Envase text,
    Capacidad float,
    Stock int,
    primary key (codc)
);
'''

crsr.execute(query)

In [11]:
query = '''
CREATE TABLE IF NOT EXISTS BARES (
    CodB TEXT,
    Nombre TEXT,
    Cif TEXT,
    Localidad TEXT,
    primary key (codb)
);
'''
crsr.execute(query)

In [12]:
query = '''
CREATE TABLE IF NOT EXISTS EMPLEADOS (
    CodE INTEGER,
    Nombre TEXT,
    Sueldo INTEGER
);
'''
crsr.execute(query)


In [13]:
query = '''
CREATE TABLE IF NOT EXISTS REPARTO (
    CodE varchar(2) not null,
    CodB varchar(3) not null,
    CodC varchar(2) not null,
    Fecha DATE not null,
    Cantidad smallint,
    primary key (code,codb,codc,fecha),
    foreign key (code) references empleados(code),
    foreign key (codb) references bares(codb),
    foreign key (codc) references cervezas(codc)
);
'''
crsr.execute(query)

In [14]:
query = '''
INSERT or replace INTO CERVEZAS (CodC, Envase, Capacidad, Stock) VALUES
('01', 'Botella', 0.2, 3600),
('02', 'Botella', 0.33, 1200),
('03', 'Lata', 0.33, 2400),
('04', 'Botella', 1, 288),
('05', 'Barril', 60, 30);
'''
crsr.execute(query)

In [15]:
query = '''
INSERT or replace INTO BARES (CodB, Nombre, Cif, Localidad) VALUES
('001', 'Stop', '11111111X', 'Villa Botijo'),
('002', 'Las Vegas', '22222222Y', 'Villa Botijo'),
('003', 'Club Social', NULL, 'Las Ranas'),
('004', 'Otra Ronda', '33333333Z', 'La Esponja');
'''
crsr.execute(query)

In [16]:
query = '''
INSERT or replace INTO EMPLEADOS (CodE, Nombre, Sueldo) VALUES
(1, 'Carlos Lopez', 120000),
(2, 'Rosa Perez', 110000),
(3, 'Luisa Garcia', 100000);
'''
crsr.execute(query)

In [17]:
query = '''
INSERT or replace INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad) VALUES
(1, '001', '01', '10/21/05', 240),
(1, '001', '02', '10/21/05', 48),
(1, '002', '03', '10/22/05', 60),
(1, '004', '05', '10/22/05', 4),
(2, '002', '03', '10/22/05', 48),
(2, '002', '05', '10/23/05', 2),
(2, '004', '01', '10/23/05', 480),
(2, '004', '02', '10/24/05', 72),
(3, '003', '03', '10/24/05', 48),
(3, '003', '04', '10/25/05', 20);
'''
crsr.execute(query)


In [18]:
query = '''
select * from cervezas
'''
sql_query(query)

,CodC,Envase,Capacidad,Stock
0,1,Botella,0.20,3600
1,2,Botella,0.33,1200
2,3,Lata,0.33,2400
3,4,Botella,1.00,288
4,5,Barril,60.00,30


In [19]:
query = '''
select * from bares
'''
sql_query(query)

,CodB,Nombre,Cif,Localidad
0,001,Stop,11111111X,Villa Botijo
1,002,Las Vegas,22222222Y,Villa Botijo
2,003,Club Social,NaN,Las Ranas
3,004,Otra Ronda,33333333Z,La Esponja


In [20]:
query = '''
select * from empleados
'''
sql_query(query)

,CodE,Nombre,Sueldo
0,1,Carlos Lopez,120000
1,2,Rosa Perez,110000
2,3,Luisa Garcia,100000


In [21]:
query = '''
select * from reparto
'''
sql_query(query)

,CodE,CodB,CodC,Fecha,Cantidad
0,1,001,01,10/21/05,240
1,1,001,02,10/21/05,48
2,1,002,03,10/22/05,60
3,1,004,05,10/22/05,4
4,2,002,03,10/22/05,48
5,2,002,05,10/23/05,2
6,2,004,01,10/23/05,480
7,2,004,02,10/24/05,72
8,3,003,03,10/24/05,48
9,3,003,04,10/25/05,20


In [22]:
# 1 Obtener el nombre de los empleados que hayan repartido al bar Stop durante la semana 
# del 17 al 23 de octubre de 2005.

query = '''
select e.nombre, b.nombre, r.fecha
from reparto as r
left join empleados as e 
on r.CodE = e.CodE
left join bares as b
on b.CodB = r.CodB
where b.nombre = "Stop"
    and r.fecha between "10/17/05" AND "10/23/05"
'''

sql_query(query)

,Nombre,Nombre,Fecha
0,Carlos Lopez,Stop,10/21/05
1,Carlos Lopez,Stop,10/21/05


In [23]:
# 2 Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo Botella y
# capacidad inferior a 1 litro, ordenados por localidad

query = '''
select b.cif, b.nombre, c.envase, c.capacidad, b.localidad
from reparto as r
left join empleados as e 
on r.CodE = e.CodE
left join bares as b
on b.CodB = r.CodB
left join cervezas as c
on c.CodC = r.CodC
where c.envase like "Botella"
and c.capacidad < 1
order by b.localidad
'''

sql_query(query)

,Cif,Nombre,Envase,Capacidad,Localidad
0,33333333Z,Otra Ronda,Botella,0.20,La Esponja
1,33333333Z,Otra Ronda,Botella,0.33,La Esponja
2,11111111X,Stop,Botella,0.20,Villa Botijo
3,11111111X,Stop,Botella,0.33,Villa Botijo


In [24]:
# 3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad)

query = '''
select b.nombre, c.envase, c.capacidad, r.fecha, r.cantidad
from reparto as r
left join empleados as e 
on r.CodE = e.CodE
left join bares as b
on b.CodB = r.CodB
left join cervezas as c
on c.CodC = r.CodC
'''

sql_query(query)

,Nombre,Envase,Capacidad,Fecha,Cantidad
0,Stop,Botella,0.20,10/21/05,240
1,Stop,Botella,0.33,10/21/05,48
2,Las Vegas,Lata,0.33,10/22/05,60
3,Otra Ronda,Barril,60.00,10/22/05,4
4,Las Vegas,Lata,0.33,10/22/05,48
5,Las Vegas,Barril,60.00,10/23/05,2
6,Otra Ronda,Botella,0.20,10/23/05,480
7,Otra Ronda,Botella,0.33,10/24/05,72
8,Club Social,Lata,0.33,10/24/05,48
9,Club Social,Botella,1.00,10/25/05,20


In [25]:
# 4. Obtener los bares a los que se les ha repartido envases de tipo botella y capacidad 0.2 o
# 0.33

query = '''
select b.nombre, c.envase, c.capacidad
from reparto as r
left join empleados as e 
on r.CodE = e.CodE
left join bares as b
on b.CodB = r.CodB
left join cervezas as c
on c.CodC = r.CodC
where c.envase like "Botella"
and c.capacidad = 0.2
or c.envase like "Botella"
and c.capacidad = 0.33
'''
sql_query(query)

,Nombre,Envase,Capacidad
0,Stop,Botella,0.20
1,Stop,Botella,0.33
2,Otra Ronda,Botella,0.20
3,Otra Ronda,Botella,0.33


In [26]:
# 5. Nombre de los empleados que han repartido a los bares "Stop" y "Las Vegas" cervezas con
# envase botella.

query = '''
select e.nombre, c.envase, b.nombre
from reparto as r
left join empleados as e 
on r.CodE = e.CodE
left join bares as b
on b.CodB = r.CodB
left join cervezas as c
on c.CodC = r.CodC
where b.nombre like "Stop"
and c.envase like "Botella"
or b.nombre like "Las Vegas"
and c.envase like "Botella"
'''
sql_query(query)

,Nombre,Envase,Nombre
0,Carlos Lopez,Botella,Stop
1,Carlos Lopez,Botella,Stop


In [31]:

query = '''
select e.nombre, c.envase, b.nombre
from reparto as r
left join empleados as e 
on r.CodE = e.CodE
left join bares as b
on b.CodB = r.CodB
left join cervezas as c
on c.CodC = r.CodC
where b.nombre in ("Stop", "Las Vegas")
and c.envase like "Botella"
'''
sql_query(query)

,Nombre,Envase,Nombre
0,Carlos Lopez,Botella,Stop
1,Carlos Lopez,Botella,Stop


In [32]:
# 6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de Villa
# Botijo.

query = '''
select e.nombre, c.envase, b.nombre
from reparto as r
left join empleados as e 
on r.CodE = e.CodE
left join bares as b
on b.CodB = r.CodB
left join cervezas as c
on c.CodC = r.CodC
where b.localidad not like "Villa Botijo"
group by e.code

'''
sql_query(query)

,Nombre,Envase,Nombre
0,Carlos Lopez,Barril,Otra Ronda
1,Rosa Perez,Botella,Otra Ronda
2,Luisa Garcia,Lata,Club Social


In [ ]:
# 7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.

query = '''
select b.nombre, b.localidad, max(r.cantidad)
from reparto as r
left join empleados as e 
on r.CodE = e.CodE
left join bares as b
on b.CodB = r.CodB
left join cervezas as c
on c.CodC = r.CodC
'''
sql_query(query)

,Nombre,Localidad,max(r.cantidad)
0,Otra Ronda,La Esponja,480


In [ ]:
query = '''
select b.Nombre, b.Localidad
from reparto as r
left join empleados as e 
on r.CodE = e.CodE
left join bares as b
on b.CodB = r.CodB
left join cervezas as c
on c.CodC = r.CodC
group by b.CodB, b.Nombre, b.Localidad
order by sum (r.Cantidad * c.Capacidad) desc
LIMIT 1;
'''
sql_query(query)

,Nombre,Localidad
0,Otra Ronda,La Esponja


In [33]:
# 7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.

query = '''
SELECT b.nombre, b.localidad, SUM(r.cantidad * c.capacidad) as Litros
FROM REPARTO as r
LEFT JOIN BARES as b
ON r.CodB = b.CodB
LEFT JOIN CERVEZAS as c
ON r.CodC = c.CodC
GROUP BY b.nombre
ORDER BY Litros DESC
LIMIT 1
'''
sql_query(query)

,Nombre,Localidad,Litros
0,Otra Ronda,La Esponja,359.76


In [ ]:
# 8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y
# capacidad menor que 1 litro.

query = '''
select b.nombre, c.envase, b.nombre
from reparto as r
left join empleados as e 
on r.CodE = e.CodE
left join bares as b
on b.CodB = r.CodB
left join cervezas as c
on c.CodC = r.CodC
where c.capacidad < 1
and c.envase like "Botella"
GROUP BY b.CodB, b.Nombre
having count (distinct r.codc) = 2;
'''
sql_query(query)

,Nombre,Envase,Nombre
0,Stop,Botella,Stop
1,Otra Ronda,Botella,Otra Ronda


In [ ]:
query = '''
select b.nombre
from bares b
left join reparto r on b.codb = r.codb
left join cervezas c on r.codc = c.codc
group by b.codb, b.nombre
having count(distinct c.codc) = (
    select count(*)
    from cervezas
    where envase = 'botella'
      and capacidad < 1
);
'''
sql_query(query)

,Nombre


In [34]:
#9. Subir un 5% el sueldo del empleado que más días haya trabajado.

# query = '''
# select *
# from reparto as r
# left join empleados as e 
# on r.CodE = e.CodE
# left join bares as b
# on b.CodB = r.CodB
# left join cervezas as c
# on c.CodC = r.CodC
# update e
# set e.sueldo = e.sueldo * 1.05
# where r.code = (
#     select r.code
#     from r
#     group by r.code
#     order by count(*) desc
#     limit 1
# );
# '''
# sql_query(query)

In [ ]:
query = '''
select r.code, e.nombre, count(*) as dias_trabajados
from reparto as r
left join empleados as e 
on r.CodE = e.CodE
group by r.code
order by dias_trabajados desc
limit 1;
'''
sql_query(query)

,CodE,Nombre,dias_trabajados
0,2,Rosa Perez,4


In [ ]:
query = '''
update e.sueldo
from reparto as r
left join empleados as e 
on r.CodE = e.CodE
group by r.code
where
select r.code, e.nombre, count(*) as dias_trabajados
from reparto as r
left join empleados as e 
on r.CodE = e.CodE
group by r.code
order by dias_trabajados desc
limit 1;
'''
sql_query(query)

,t.Sueldo * 1.05
0,115500.0


In [ ]:
query = '''

'''
crsr.execute(query)

In [ ]:
query = '''

'''
sql_query(query)

,CodE,Nombre,Sueldo
0,2,Rosa Perez,115500
